The path should be set at the 'Data' folder

In [13]:
cd DRAFT/Data

/research/phd/phd2k22/cse/rudra.dhar/DRAFT/Data


/research/phd/phd2k22/cse/rudra.dhar/miniconda3/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


## Aggregate

First process ADR metadata from JSON files in the 'repositories' folder and generates a CSV summary (along with url). <br>
Also counts occurrences of each ADR template type.

In [2]:
import json
import os
import csv

In [ ]:


folder = 'repositories'
template_count = {}

# Ensure output directory exists
os.makedirs('ADR-data', exist_ok=True)

# Create new csv file
with open('ADR-data/data.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
    writer.writerow(['URL', 'ADR Folders', 'Number of ADR Files', 'Names of ADR Folders', 'Names of ADR Files'])
    
    for filename in os.listdir(folder):
        with open(os.path.join(folder, filename)) as json_file:
            data = json.load(json_file)

            names_of_folders = '| '.join(data['adrDirectories'])
            names_of_files = '| '.join((adr['adrDirectory'] + '/' + adr['path']) for adr in data['adrFiles'])

            for file in data['adrFiles']:
                if template_count.get(file['template']) is None:
                    template_count[file['template']] = 1
                else:
                    template_count[file['template']] += 1

            writer.writerow([
                data['repositoryUrl'], 
                data['numAdrDirectories'], 
                data['numAdrFiles'], 
                names_of_folders, 
                names_of_files
            ])
        
    
print(template_count)


{'Nygard': 4146, 'unknown': 1613, 'Madr': 599, 'Alexandrian': 4}


## Scrape

Next scrape the ADRs using Github api, and store them in the folder 'all_ADRs'. Use the csv created above for this process. <br>
Since it take huge time, its kept in 'scrape.py'

## Dataset creation

Next make a jsonl of ADRS from the scraped ADRs. <br>
We filter out ADRs Title, Body, Context and Decision. <br>

For getting the Title, we 1st look into the ADR's top with some string matching and regex; if we dont find the Title there, then we use the file name. <br>
Next we filter out the Decision, and then the Context. <br>
If a Decision is found and Context is not found, we use the part above the Decision as context.

In [81]:
# Install required libraries if you don't have them already
# !pip install transformers pandas torch

import os
import re
import json
import tiktoken
# from transformers import AutoTokenizer

#tokenizer initialization
CACHE_DIR = "../../cache"
os.environ["TIKTOKEN_CACHE_DIR"] = CACHE_DIR
encoding = tiktoken.encoding_for_model("gpt-4")

In [87]:
# --- HEADINGS FOR EXTRACTION ---
# List of headings that signify the "Context" section
CONTEXT_HEADINGS = [
    'Context',
    'Context and Problem Statement',
    'Decision Drivers',
    'Pros and Cons of the Options',
    'Problem',
]

# List of headings that signify the "Decision" section
DECISION_HEADINGS = [
    'Decision',
    'Decision Outcome',
    'Decisions',
    'Proposed Solutions'
]


# --- HELPER FUNCTIONS ---

def count_tokens(text: str) -> int:
    """Counts the number of tokens in a given text string."""
    if not text:
        return 0
    # Replace with your actual encoding logic
    # return len(encoding.encode(text))
    return len(text.split()) # Placeholder if tiktoken is not set up


# --- NEW: Heading Parser ---
def get_heading_info(lines: list[str], i: int) -> tuple[int, str]:
    """
    Checks if a line is a heading and returns its level and text.
    - ATX Headings: # (H1), ## (H2), etc.
    - Setext Headings: Title\n=== (H1), Title\n--- (H2)
    
    This version also strips common markdown and punctuation characters
    like *, :, and _ from the ends of the heading text.
    
    Returns:
        A tuple (level, text). `level` is 0 if the line is not a heading.
    """
    if i >= len(lines):
        return 0, ""

    line = lines[i].strip()

    # Check for ATX style headings (e.g., "## *Title:*")
    if line.startswith('#'):
        level = 0
        text = ""
        for char in line:
            if char == '#':
                level += 1
            else:
                # Extract the raw text part
                raw_text = line[level:].strip()
                # Clean the text by stripping unwanted characters from the ends
                text = raw_text.strip('*:_ ')
                break
        return level, text

    # Check for Setext style headings (e.g., "*Title:*\n---")
    if i + 1 < len(lines):
        text = line
        underline = lines[i+1].strip()
        if text and underline:
            # Clean the text by stripping unwanted characters from the ends
            cleaned_text = text.strip('*:_ ')
            if all(ch == '=' for ch in underline):
                return 1, cleaned_text
            if all(ch == '-' for ch in underline):
                return 2, cleaned_text

    return 0, ""


# --- Specialized Parsers ---

def extract_title(lines: list[str], file_path: str) -> tuple[str, list[str]]:
    """
    Extracts the title from the ADR.
    (This function remains unchanged).
    """
    filename = os.path.splitext(os.path.basename(file_path))[0]
    if not lines:
        return filename, []
    first_idx = next((i for i, ln in enumerate(lines) if ln.strip()), None)
    if first_idx is None:
        return filename, []
    first_line = lines[first_idx].strip()
    title = None
    remaining = lines[first_idx+1:]
    if first_line == "---":
        i = first_idx + 1
        meta_lines = []
        while i < len(lines) and lines[i].strip() != "---":
            meta_lines.append(lines[i].strip())
            i += 1
        remaining = lines[i+1:] if i < len(lines) else lines[first_idx+1:]
        for ml in meta_lines:
            m = re.match(r"^\s*title\s*:\s*(.+)$", ml, re.IGNORECASE)
            if m:
                title = m.group(1).strip().strip("'\"")
                break
    if not title and first_line.startswith("#"):
        candidate = first_line.lstrip("#").strip()
        if candidate:
            title = candidate
        remaining = lines[first_idx+1:]
    if not title:
        m = re.match(r"ADR[-\s]?\d+\s*[:\-]\s*(.+)", first_line, re.IGNORECASE)
        if m:
            candidate = m.group(1).strip().strip("'\"")
            if candidate:
                title = candidate
            remaining = lines[first_idx+1:]
    if not title:
        title = filename
        remaining = lines[first_idx:]
    bad_titles = {"title", "context", "adrs", "adr"}
    
    if (
        title.lower() in bad_titles
        or re.fullmatch(r"adr-\d+", title, re.IGNORECASE)
        or re.fullmatch(r"\d+", title)
        or len(title) < 3
    ):
        title = filename
    return title, remaining


# --- UPDATED: Level-Aware Section Extractor ---
def extract_section(lines: list[str], target_headings: list[str]) -> tuple[str, int, int]:
    """
    Finds a section and extracts its content, aware of heading levels.
    A section (e.g., H2) ends only upon finding another heading of the same
    or higher level (H2 or H1), ignoring subheadings (H3, H4, etc.).
    
    This version checks if a heading *starts with* one of the target headings.
    """
    start_index = -1
    target_level = 0
    heading_lines_to_skip = 0
    normalized_targets = [h.lower() for h in target_headings]

    # 1. Find the start of the target section
    for i in range(len(lines)):
        level, text = get_heading_info(lines, i)

        # Check if the heading text starts with any of the targets
        if level > 0:
            lower_text = text.lower()
            # Iterate through the normalized targets to check for a prefix match
            for target in normalized_targets:
                if lower_text.startswith(target):
                    start_index = i
                    target_level = level
                    # Determine lines to skip (1 for ATX, 2 for Setext)
                    heading_lines_to_skip = 2 if not lines[i].strip().startswith('#') else 1
                    break  # Exit the inner loop over targets
            
            if start_index != -1:
                break # Exit the outer loop over lines once a match is found

    if start_index == -1:
        return "", -1, -1

    # 2. Capture content until the next heading of same or higher level
    content_lines = []
    content_start_index = start_index + heading_lines_to_skip
    end_index = len(lines)

    for i in range(content_start_index, len(lines)):
        level, _ = get_heading_info(lines, i)
        # Stop if we hit a new heading that is NOT a subheading of our target
        if 0 < level <= target_level:
            end_index = i
            break
        content_lines.append(lines[i])

    return "".join(content_lines), start_index, end_index


# --- Orchestrator Function ---
def process_adr_file(file_path: str):
    """
    Processes a single ADR file to extract Title, Body, Context, and Decision.
    (This function remains unchanged).
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            all_lines = f.readlines()
    except Exception as e:
        print(f"Could not read file {file_path}: {e}")
        return None

    if not all_lines:
        print(f"File {file_path} is empty.")
        return None

    name_of_adr, body_lines = extract_title(all_lines, file_path)
    body_of_adr = "".join(body_lines)

    decision_text, decision_start_idx, decision_end_idx = extract_section(
        body_lines, DECISION_HEADINGS
    )

    if not decision_text.strip():
        # print(f"No decision extracted from {file_path}")
        return None

    context_text, _, __ = extract_section(body_lines, CONTEXT_HEADINGS)

    if not context_text.strip():
        context_fallback_lines = (
            body_lines[:decision_start_idx]
        )
        context_text = "".join(context_fallback_lines)

    adr_data = {
        "Title": name_of_adr,
        "Body": body_of_adr,
        "Context": context_text.strip(),
        "Decision": decision_text.strip(),
        "tokenBody": count_tokens(body_of_adr),
        "tokenContext": count_tokens(context_text),
        "tokenDecision": count_tokens(decision_text),
        "Path": file_path,
    }
    return adr_data

In [88]:
# --- MAIN EXECUTION ---

PARENT_DIR = 'all_ADRs' 
OUTPUT_FILE = 'ADR-data/adrs_output.jsonl'

# Counter for processed files
processed_count = 0
skipped_count = 0

print(f"Starting ADR processing from directory: {PARENT_DIR}")

# Open the output file for writing
with open(OUTPUT_FILE, 'w', encoding='utf-8') as outfile:
    # Get all subfolders in the parent directory
    folders = [f for f in os.listdir(PARENT_DIR) if os.path.isdir(os.path.join(PARENT_DIR, f))]
    
    for folder in folders:
        folder_path = os.path.join(PARENT_DIR, folder)
        files = os.listdir(folder_path)
        
        for file in files:
            # We assume ADRs are markdown files
            if not file.endswith(('.md', '.mdx')):
                continue

            file_path = os.path.join(folder_path, file)
            
            # Process the file to get the structured data
            adr_data = process_adr_file(file_path)
            
            # If the file was processed successfully, write it to the jsonl file
            if adr_data:
                # Convert dictionary to a JSON string and write it as a new line
                outfile.write(json.dumps(adr_data) + '\n')
                processed_count += 1
            else:
                skipped_count += 1

print("\n--- Processing Complete! ---")
print(f"✅ Successfully processed and saved: {processed_count} ADRs")
print(f"❌ Skipped (missing context/decision): {skipped_count} files")
print(f"📄 Output saved to: {OUTPUT_FILE}")

Starting ADR processing from directory: all_ADRs
File all_ADRs\roster\frontend-002.code-in-typescript.md is empty.
File all_ADRs\staff-infrastructure-monitoring\0003-use-zabbix-for-physical-network.md is empty.

--- Processing Complete! ---
✅ Successfully processed and saved: 5766 ADRs
❌ Skipped (missing context/decision): 371 files
📄 Output saved to: ADR-data/adrs_output.jsonl


## Filtering

1st we get a stats on tokens in Body, Context and decision. <br>

Then we filter out based on
 - Token count i.e. keep ADR within a range
 - Non English i.e. high amount of non Ascii characters
 - High amount of url

In [89]:
import json
import statistics
import string
import re

Infile = 'ADR-data/adrs_output.jsonl'
Outfile = 'ADR-data/filtered_adrs_output.jsonl'

In [90]:
def get_token_stats(jsonl_file):
    token_body, token_context, token_decision = [], [], []

    # Read JSONL file line by line
    with open(jsonl_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():  # skip empty lines
                record = json.loads(line)
                token_body.append(record.get("tokenBody", 0))
                token_context.append(record.get("tokenContext", 0))
                token_decision.append(record.get("tokenDecision", 0))

    def stats(values):
        if not values:
            return {"min": 0, "avg": 0, "max": 0, "median": 0, "mode": None}
        try:
            mode_val = statistics.mode(values)
        except statistics.StatisticsError:
            mode_val = None  # no unique mode
        return {
            "min": min(values),
            "avg": sum(values) / len(values),
            "max": max(values),
            "median": statistics.median(values),
            "mode": mode_val,
        }

    return {
        "tokenBody": stats(token_body),
        "tokenContext": stats(token_context),
        "tokenDecision": stats(token_decision),
    }


results = get_token_stats(Infile)
for k, v in results.items():
    print(
        f"{k}: min={v['min']}, avg={v['avg']:.2f}, "
        f"max={v['max']}, median={v['median']}, mode={v['mode']}"
    )


tokenBody: min=12, avg=354.32, max=5850, median=211.0, mode=48
tokenContext: min=0, avg=106.41, max=2236, median=59.0, mode=11
tokenDecision: min=1, avg=100.62, max=5704, median=38.0, mode=11


In [91]:
URL_RE = re.compile(r'https?://[^\s]+|www\.[^\s]+')

def clean_text(text: str) -> str:
    if not text:
        return ""
    # Replace literal "\n" and actual newlines with spaces
    text = text.replace("\\n", " ")
    text = text.replace("\n", " ")
    return text

def clean_token(tok: str) -> str:
    return tok.strip(string.punctuation)

def url_ratio(text: str) -> float:
    """Return fraction of tokens that look like URLs."""
    text = clean_text(text)
    tokens = [clean_token(t) for t in text.split()]
    if not tokens:
        return 0.0
    url_count = sum(1 for t in tokens if URL_RE.fullmatch(t))
    return url_count / len(tokens)
    
def is_mostly_ascii(text, threshold=0.9):
    if not text:
        return True
    ascii_chars = sum(c in string.printable for c in text)
    ratio = ascii_chars / len(text)
    return ratio >= threshold

def filter_jsonl(input_file, output_file):
    min_body, max_body = 10, 1000
    min_context, max_context = 10, 500
    min_decision, max_decision = 2, 500

    total = kept = dropped = 0

    with open(input_file, "r", encoding="utf-8") as infile, \
         open(output_file, "w", encoding="utf-8") as outfile:
        
        for line in infile:
            if not line.strip():
                continue

            total += 1
            record = json.loads(line)

            body = record.get("Body", "")
            context = record.get("Context", "")
            decision = record.get("Decision", "")

            # token filters
            token_ok = (
                min_body <= int(record.get("tokenBody", 0)) <= max_body and
                min_context <= int(record.get("tokenContext", 0)) <= max_context and
                min_decision <= int(record.get("tokenDecision", 0)) <= max_decision
            )

            # language filter
            text_ok = is_mostly_ascii(body)

            # URL ratio filter (>= 0.5 → drop)
            url_ok = (
                url_ratio(context) < 0.2 and
                url_ratio(decision) < 0.2
            )
            # print('token: ', token_ok, '; text: ', text_ok, '; url: ', url_ok)

            if token_ok and text_ok and url_ok:
                outfile.write(json.dumps(record, ensure_ascii=False) + "\n")
                kept += 1
            else:
                # print(f"Dropping record due to filters: {record.get('Path', 'unknown path')}")
                dropped += 1

    print("📊 Filtering Summary")
    print(f"  Total records: {total}")
    print(f"  Kept:          {kept}")
    print(f"  Dropped:       {dropped}")
    print(f"  Output file:   {output_file}")


input_path = Infile
output_path = Outfile
filter_jsonl(input_path, output_path)

📊 Filtering Summary
  Total records: 5766
  Kept:          5042
  Dropped:       724
  Output file:   ADR-data/filtered_adrs_output.jsonl


In [92]:
results = get_token_stats(Outfile)
for k, v in results.items():
    print(
        f"{k}: min={v['min']}, avg={v['avg']:.2f}, "
        f"max={v['max']}, median={v['median']}, mode={v['mode']}"
    )

tokenBody: min=18, avg=258.03, max=995, median=196.0, mode=48
tokenContext: min=10, avg=85.33, max=497, median=55.0, mode=11
tokenDecision: min=2, avg=65.52, max=500, median=36.0, mode=11


## De-duplication

De-Duplication is done based of cosine similarity (threshold = 0.98) of the **sentence-bert** representation of the **Body** of an ADR with 'all-MiniLM-L6-v2' LLM

References:<br>
sentencebert - https://arxiv.org/pdf/1908.10084 <br>
model - https://www.atlantis-press.com/proceedings/iciaai-24/126004096

In [14]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import time
import json

cache_dir = "../../cache"
model = SentenceTransformer("all-MiniLM-L6-v2", cache_folder=cache_dir)

infile = "ADR-data/filtered_adrs_output.jsonl"
outfile = "ADR-data/adrs.jsonl"
threshold = 0.98  # similarity threshold for duplicates (tune as needed)
dupes_file = "ADR-data/removed_duplicates.jsonl"  # store removed ones

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [27]:
# Step 1. Load JSONL
records = []
bodies = []
context_decisions = []

with open(infile, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        records.append(record)
        bodies.append(record["Body"])
        # Combine context and decision fields
        context_decision = f"{record.get('Context', '')} {record.get('Decision', '')}"
        context_decisions.append(context_decision)

# records, bodies, context_decisions = records[:1000], bodies[:1000], context_decisions[:1000]

# Step 2. Encode with Sentence-BERT
start_time = time.time()
print("Encoding bodies...")
body_embeddings = model.encode(bodies, convert_to_numpy=True, normalize_embeddings=True)

print("Encoding context + decisions...")
context_decision_embeddings = model.encode(context_decisions, convert_to_numpy=True, normalize_embeddings=True)

encoding_time = time.time()
print(f"Encoding time: {encoding_time - start_time:.2f} seconds")

# Step 3. Pairwise Cosine Similarity for both fields
print("Computing body similarities...")
body_similarity_matrix = cosine_similarity(body_embeddings)

print("Computing context+decision similarities...")
context_decision_similarity_matrix = cosine_similarity(context_decision_embeddings)

print(f"Matching time: {time.time() - encoding_time:.2f} seconds")

# Step 4. Enhanced Deduplication
unique_records = []
duplicates = []
seen = set()

for i in range(len(records)):
    if i in seen:
        continue

    # keep this record, add primary key
    record_with_id = records[i].copy()
    record_with_id["PrimaryKey"] = len(unique_records) + 1
    unique_records.append(record_with_id)

    base_path = records[i]["Path"]

    # mark duplicates based on EITHER body similarity OR context+decision similarity
    for j in range(i + 1, len(records)):
        is_body_duplicate = body_similarity_matrix[i][j] >= threshold
        is_context_decision_duplicate = context_decision_similarity_matrix[i][j] >= threshold
        
        if is_body_duplicate or is_context_decision_duplicate:
            seen.add(j)
            
            # Track which field(s) caused the duplication
            duplicate_reasons = []
            if is_body_duplicate:
                duplicate_reasons.append(f"body_sim={body_similarity_matrix[i][j]:.3f}")
            if is_context_decision_duplicate:
                duplicate_reasons.append(f"context_decision_sim={context_decision_similarity_matrix[i][j]:.3f}")
            
            duplicates.append({
                "Path": records[j]["Path"],
                "DuplicateOf": base_path,
                "Reason": "; ".join(duplicate_reasons)
            })

print(f"Original records: {len(records)}")
print(f"Unique records: {len(unique_records)}")
print(f"Duplicates found: {len(duplicates)}")

# Step 5. Save outputs
with open(outfile, "w", encoding="utf-8") as f:
    for rec in unique_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

with open(dupes_file, "w", encoding="utf-8") as f:
    for rec in duplicates:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

Encoding bodies...
Encoding context + decisions...
Encoding time: 6.76 seconds
Computing body similarities...
Computing context+decision similarities...
Matching time: 0.14 seconds
Original records: 5042
Unique records: 4344
Duplicates found: 769


## Train-Val-Test split

We did Train-Val-Test split of 70-10-20.<br>
We kept test set a bit larger as 2 of the experiments, prompting, and RAFG will be done just on the test set.

In [29]:
import json
import random

# Input file
infile = "ADR-data/adrs.jsonl"

# Output files
train_file = "ADR-data/train.jsonl"
val_file = "ADR-data/val.jsonl"
test_file = "ADR-data/test.jsonl"

# Step 1. Load JSONL
records = []
with open(infile, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        records.append(record)

print("Total records loaded:", len(records))

# Step 2. Shuffle for randomness
random.shuffle(records)

# Step 3. Compute split sizes
n_total = len(records)
n_train = int(0.7 * n_total)
n_val = int(0.1 * n_total)
n_test = n_total - n_train - n_val  # ensures no rounding loss

train_records = records[:n_train]
val_records = records[n_train:n_train + n_val]
test_records = records[n_train + n_val:]

print("Train:", len(train_records))
print("Validation:", len(val_records))
print("Test:", len(test_records))

# Step 4. Save outputs
def save_jsonl(filename, data):
    with open(filename, "w", encoding="utf-8") as f:
        for rec in data:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

save_jsonl(train_file, train_records)
save_jsonl(val_file, val_records)
save_jsonl(test_file, test_records)

print("✅ Train/Val/Test splits saved in ADR-data/")

Total records loaded: 4344
Train: 3040
Validation: 434
Test: 870
✅ Train/Val/Test splits saved in ADR-data/
